In [ ]:
"""Generador deterministico de proyectos integradores IoT para Google Colab.
"""

from __future__ import annotations

import hashlib
import html
import re
import subprocess
import sys
from dataclasses import dataclass
from getpass import getpass
from pathlib import Path
from typing import Any, Sequence


COURSE_NAME = "Internet de las Cosas"
COURSE_VERSION = "IOT-2026-B1-v1"
BOARD_NAME = "ESP32 DevKit v1"
BOARD_WOKWI_ID = "wokwi-esp32-devkit-v1"


def _ensure_reportlab() -> None:
    """Install ReportLab only when the current runtime does not provide it."""
    try:
        import reportlab  # noqa: F401
    except ModuleNotFoundError:
        print("Instalando la biblioteca necesaria para crear el PDF...")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "--quiet", "reportlab"]
        )


_ensure_reportlab()

from reportlab.lib import colors  # noqa: E402
from reportlab.lib.enums import TA_CENTER, TA_LEFT  # noqa: E402
from reportlab.lib.pagesizes import letter  # noqa: E402
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet  # noqa: E402
from reportlab.lib.units import cm  # noqa: E402
from reportlab.pdfbase.pdfmetrics import stringWidth  # noqa: E402
from reportlab.platypus import (  # noqa: E402
    CondPageBreak,
    KeepTogether,
    LongTable,
    Paragraph,
    SimpleDocTemplate,
    Spacer,
    Table,
    TableStyle,
)


NAVY = colors.HexColor("#15324A")
BLUE = colors.HexColor("#245B78")
ORANGE = colors.HexColor("#E66A2C")
PALE_BLUE = colors.HexColor("#EAF2F7")
PALE_ORANGE = colors.HexColor("#FFF2E8")
LIGHT_GRAY = colors.HexColor("#F3F5F7")
MID_GRAY = colors.HexColor("#6B7280")
WHITE = colors.white
BLACK = colors.HexColor("#17202A")


SENSORS: dict[str, dict[str, str]] = {
    "dht22": {
        "name": "DHT22",
        "wokwi_id": "wokwi-dht22",
        "interface": "Digital",
    },
    "hc_sr04": {
        "name": "HC-SR04",
        "wokwi_id": "wokwi-hc-sr04",
        "interface": "Digital por pulsos",
    },
    "ldr": {
        "name": "Modulo fotoresistor LDR",
        "wokwi_id": "wokwi-photoresistor-sensor",
        "interface": "Analogica y digital",
    },
    "pir": {
        "name": "Sensor de movimiento PIR",
        "wokwi_id": "wokwi-pir-motion-sensor",
        "interface": "Digital",
    },
    "mq2": {
        "name": "Sensor de gas MQ2",
        "wokwi_id": "wokwi-gas-sensor",
        "interface": "Analogica",
    },
    "mpu6050": {
        "name": "MPU6050",
        "wokwi_id": "wokwi-mpu6050",
        "interface": "I2C",
    },
    "hx711": {
        "name": "Celda de carga con HX711",
        "wokwi_id": "wokwi-hx711",
        "interface": "Digital",
    },
    "mfrc522": {
        "name": "Lector RFID MFRC522",
        "wokwi_id": "board-mfrc522",
        "interface": "SPI",
    },
    "ds18b20": {
        "name": "DS18B20",
        "wokwi_id": "wokwi-ds18b20",
        "interface": "OneWire",
    },
    "bmp180": {
        "name": "BMP180",
        "wokwi_id": "wokwi-bmp180",
        "interface": "I2C",
    },
}


ACTUATORS: dict[str, dict[str, str]] = {
    "relay": {
        "name": "Modulo rele",
        "wokwi_id": "wokwi-relay-module",
        "commands": "AUTO, ACTIVAR y DESACTIVAR",
        "safe_state": "salida desactivada",
    },
    "servo": {
        "name": "Servomotor",
        "wokwi_id": "wokwi-servo",
        "commands": "AUTO, ABRIR y CERRAR",
        "safe_state": "posicion de 0 grados",
    },
    "buzzer": {
        "name": "Buzzer piezoelectrico",
        "wokwi_id": "wokwi-buzzer",
        "commands": "AUTO, ARMAR y SILENCIAR",
        "safe_state": "apagado, excepto ante una alarma local confirmada",
    },
    "rgb": {
        "name": "LED RGB",
        "wokwi_id": "wokwi-rgb-led",
        "commands": "AUTO, NORMAL, ALERTA y APAGAR",
        "safe_state": "indicacion amarilla intermitente",
    },
    "neopixel": {
        "name": "Anillo NeoPixel",
        "wokwi_id": "wokwi-led-ring",
        "commands": "AUTO, NORMAL, ALERTA y APAGAR",
        "safe_state": "indicacion amarilla intermitente",
    },
}


DISPLAYS: tuple[dict[str, str], ...] = (
    {
        "name": "Pantalla OLED SSD1306",
        "wokwi_id": "board-ssd1306",
        "requirement": "Mostrar la medicion, el modo de operacion y el estado de red.",
    },
    {
        "name": "Pantalla LCD 1602 con I2C",
        "wokwi_id": "wokwi-lcd1602",
        "requirement": "Mostrar la medicion, el modo de operacion y el estado de alerta.",
    },
    {
        "name": "LED de estado",
        "wokwi_id": "wokwi-led",
        "requirement": "Indicar conexion, operacion normal y fallo con patrones diferentes.",
    },
)


@dataclass(frozen=True)
class ProjectTemplate:
    key: str
    title: str
    sensor: str
    contexts: tuple[str, ...]
    variables: tuple[str, ...]
    unit: str
    threshold_min: int
    threshold_max: int
    threshold_step: int
    condition: str
    actuators: tuple[str, ...]
    purpose: str
    low_test: str
    high_test: str


TEMPLATES: tuple[ProjectTemplate, ...] = (
    ProjectTemplate(
        key="temperature_dht22",
        title="Supervision termica",
        sensor="dht22",
        contexts=(
            "un invernadero urbano",
            "una sala de equipos",
            "una bodega de insumos",
            "un aula de estudio",
        ),
        variables=("temperatura",),
        unit="grados C",
        threshold_min=23,
        threshold_max=34,
        threshold_step=1,
        condition="activar la respuesta cuando la temperatura sea mayor o igual al umbral",
        actuators=("relay", "rgb", "buzzer"),
        purpose="Evitar condiciones termicas inadecuadas y conservar un registro de eventos.",
        low_test="Simular una temperatura cinco grados por debajo del umbral.",
        high_test="Simular una temperatura cuatro grados por encima del umbral.",
    ),
    ProjectTemplate(
        key="humidity_dht22",
        title="Control de humedad ambiental",
        sensor="dht22",
        contexts=(
            "un archivo documental",
            "un cultivo interior",
            "una zona de almacenamiento",
            "un laboratorio academico",
        ),
        variables=("humedad relativa",),
        unit="% HR",
        threshold_min=55,
        threshold_max=82,
        threshold_step=1,
        condition="activar la respuesta cuando la humedad sea mayor o igual al umbral",
        actuators=("relay", "rgb", "buzzer"),
        purpose="Detectar humedad excesiva y comunicar oportunamente el cambio de estado.",
        low_test="Simular humedad estable diez puntos por debajo del umbral.",
        high_test="Simular humedad sostenida cinco puntos por encima del umbral.",
    ),
    ProjectTemplate(
        key="tank_distance",
        title="Supervision de nivel por distancia",
        sensor="hc_sr04",
        contexts=(
            "un tanque de agua lluvia",
            "un deposito de alimento seco",
            "un contenedor de reciclaje",
            "un sistema de almacenamiento de insumos",
        ),
        variables=("distancia libre hasta la superficie",),
        unit="cm",
        threshold_min=20,
        threshold_max=120,
        threshold_step=5,
        condition="activar la respuesta cuando la distancia sea menor o igual al umbral",
        actuators=("servo", "relay", "buzzer"),
        purpose="Estimar el nivel disponible y reaccionar ante una condicion limite.",
        low_test="Simular una distancia diez centimetros menor que el umbral.",
        high_test="Simular una distancia treinta centimetros mayor que el umbral.",
    ),
    ProjectTemplate(
        key="lighting",
        title="Iluminacion adaptativa",
        sensor="ldr",
        contexts=(
            "un corredor de circulacion",
            "una zona de trabajo",
            "un cultivo interior",
            "una vitrina de exhibicion",
        ),
        variables=("iluminacion",),
        unit="lux",
        threshold_min=120,
        threshold_max=650,
        threshold_step=10,
        condition="activar la iluminacion cuando el valor sea menor o igual al umbral",
        actuators=("relay", "rgb", "neopixel"),
        purpose="Ajustar la respuesta luminosa de acuerdo con las condiciones del entorno.",
        low_test="Simular oscuridad con un valor inferior a la mitad del umbral.",
        high_test="Simular iluminacion intensa con un valor superior al doble del umbral.",
    ),
    ProjectTemplate(
        key="motion",
        title="Deteccion de presencia",
        sensor="pir",
        contexts=(
            "una zona de acceso restringido",
            "un espacio de almacenamiento",
            "un aula fuera de horario",
            "un puesto de trabajo desatendido",
        ),
        variables=("presencia",),
        unit="segundos",
        threshold_min=3,
        threshold_max=15,
        threshold_step=1,
        condition="confirmar el evento cuando la presencia se mantenga durante el tiempo asignado",
        actuators=("buzzer", "servo", "rgb"),
        purpose="Detectar presencia, reducir falsas alarmas y registrar eventos confirmados.",
        low_test="Generar un evento mas corto que el tiempo de confirmacion.",
        high_test="Generar un evento que supere el tiempo de confirmacion.",
    ),
    ProjectTemplate(
        key="gas",
        title="Alerta por gases combustibles",
        sensor="mq2",
        contexts=(
            "una cocina experimental",
            "un cuarto tecnico",
            "una zona de almacenamiento de combustibles",
            "un laboratorio de prototipado",
        ),
        variables=("nivel analogico de gas",),
        unit="unidades ADC",
        threshold_min=900,
        threshold_max=2800,
        threshold_step=50,
        condition="activar la alarma cuando la lectura sea mayor o igual al umbral",
        actuators=("buzzer", "relay", "rgb"),
        purpose="Advertir una condicion simulada de riesgo sin presentarla como instrumento certificado.",
        low_test="Simular una lectura estable doscientas unidades por debajo del umbral.",
        high_test="Simular una lectura quinientas unidades por encima del umbral.",
    ),
    ProjectTemplate(
        key="movement",
        title="Deteccion de inclinacion anormal",
        sensor="mpu6050",
        contexts=(
            "un paquete fragil en transporte",
            "un equipo movil",
            "una estructura experimental",
            "un contenedor sensible a inclinaciones",
        ),
        variables=("angulo de inclinacion",),
        unit="grados",
        threshold_min=12,
        threshold_max=55,
        threshold_step=1,
        condition="activar la respuesta cuando el valor absoluto del angulo supere el umbral",
        actuators=("buzzer", "rgb", "servo"),
        purpose="Detectar movimientos que puedan comprometer la operacion o el contenido supervisado.",
        low_test="Simular una inclinacion estable inferior al umbral.",
        high_test="Simular una inclinacion que supere el umbral durante tres mediciones.",
    ),
    ProjectTemplate(
        key="weight",
        title="Control de inventario por peso",
        sensor="hx711",
        contexts=(
            "un dispensador de alimento",
            "un estante de suministros",
            "un contenedor de materia prima",
            "una estacion de empaque",
        ),
        variables=("peso disponible",),
        unit="g",
        threshold_min=500,
        threshold_max=3500,
        threshold_step=100,
        condition="activar la respuesta cuando el peso sea menor o igual al umbral",
        actuators=("servo", "buzzer", "rgb"),
        purpose="Detectar existencias bajas y generar una notificacion reproducible.",
        low_test="Simular un peso cien gramos por debajo del umbral.",
        high_test="Simular un peso superior al doble del umbral.",
    ),
    ProjectTemplate(
        key="access",
        title="Control de acceso mediante RFID",
        sensor="mfrc522",
        contexts=(
            "un gabinete de herramientas",
            "una zona de estudio",
            "un deposito de equipos",
            "un modulo de prestamo de materiales",
        ),
        variables=("identificador RFID",),
        unit="segundos de bloqueo",
        threshold_min=5,
        threshold_max=30,
        threshold_step=1,
        condition="autorizar unicamente las credenciales asignadas y bloquear temporalmente tras tres rechazos",
        actuators=("servo", "relay", "buzzer"),
        purpose="Gestionar accesos simulados y conservar evidencia de autorizaciones y rechazos.",
        low_test="Presentar una credencial no autorizada una sola vez.",
        high_test="Presentar tres credenciales no autorizadas consecutivas y comprobar el bloqueo.",
    ),
    ProjectTemplate(
        key="temperature_ds18b20",
        title="Vigilancia de temperatura de proceso",
        sensor="ds18b20",
        contexts=(
            "una cadena de frio simulada",
            "un recipiente de proceso",
            "una cava de almacenamiento",
            "un gabinete electronico",
        ),
        variables=("temperatura",),
        unit="grados C",
        threshold_min=8,
        threshold_max=38,
        threshold_step=1,
        condition="activar la respuesta cuando la temperatura sea mayor o igual al umbral",
        actuators=("relay", "buzzer", "rgb"),
        purpose="Detectar desviaciones termicas y conservar el historial de alarmas.",
        low_test="Simular una temperatura estable cuatro grados por debajo del umbral.",
        high_test="Simular una temperatura tres grados por encima del umbral.",
    ),
    ProjectTemplate(
        key="pressure",
        title="Supervision de presion atmosferica",
        sensor="bmp180",
        contexts=(
            "una estacion ambiental escolar",
            "un prototipo de alerta meteorologica",
            "un laboratorio de observacion ambiental",
            "un sistema de registro de condiciones exteriores",
        ),
        variables=("presion atmosferica",),
        unit="hPa",
        threshold_min=970,
        threshold_max=1025,
        threshold_step=1,
        condition="activar la respuesta cuando la presion sea menor o igual al umbral",
        actuators=("buzzer", "rgb", "neopixel"),
        purpose="Registrar cambios de presion y emitir una alerta ante el escenario configurado.",
        low_test="Simular una presion diez hPa por debajo del umbral.",
        high_test="Simular una presion quince hPa por encima del umbral.",
    ),
)


class DeterministicSelector:
    """Derive independent deterministic values from a public seed."""

    def __init__(self, cedula: str) -> None:
        self._base = f"{COURSE_VERSION}|{cedula}".encode("utf-8")

    def integer(self, label: str) -> int:
        digest = hashlib.sha256(self._base + b"|" + label.encode("utf-8")).digest()
        return int.from_bytes(digest, "big")

    def choice(self, label: str, values: Sequence[Any]) -> Any:
        if not values:
            raise ValueError(f"No hay opciones para {label}")
        return values[self.integer(label) % len(values)]

    def stepped(self, label: str, minimum: int, maximum: int, step: int) -> int:
        count = ((maximum - minimum) // step) + 1
        return minimum + (self.integer(label) % count) * step


def normalize_cedula(raw_value: str) -> str:
    cedula = re.sub(r"\D", "", raw_value or "")
    if len(cedula) < 5 or len(cedula) > 15:
        raise ValueError("La cedula debe contener entre 5 y 15 digitos.")
    return cedula


def masked_cedula(cedula: str) -> str:
    return "*" * max(0, len(cedula) - 4) + cedula[-4:]


def build_assignment(cedula: str) -> dict[str, Any]:
    selector = DeterministicSelector(cedula)
    template: ProjectTemplate = selector.choice("template", TEMPLATES)
    sensor = SENSORS[template.sensor]
    actuator_key: str = selector.choice("actuator", template.actuators)
    actuator = ACTUATORS[actuator_key]
    display = selector.choice("display", DISPLAYS)
    context = selector.choice("context", template.contexts)
    variable = selector.choice("variable", template.variables)
    threshold = selector.stepped(
        "threshold",
        template.threshold_min,
        template.threshold_max,
        template.threshold_step,
    )
    hysteresis = selector.stepped("hysteresis", 2, 12, 1)
    sample_ms = selector.stepped("sample_ms", 750, 3000, 50)
    publish_seconds = selector.stepped("publish_seconds", 5, 30, 1)
    retry_seconds = selector.stepped("retry_seconds", 3, 15, 1)
    alert_confirmations = selector.stepped("alert_confirmations", 2, 6, 1)

    full_digest = hashlib.sha256(
        f"{COURSE_VERSION}|{cedula}".encode("utf-8")
    ).hexdigest().upper()
    project_code = f"IOT-{full_digest[:10]}"
    topic_root = f"iot/{full_digest[:10].lower()}"
    authorized_uid = " ".join(
        full_digest[index : index + 2] for index in range(10, 18, 2)
    )

    assignment = {
        "project_code": project_code,
        "masked_cedula": masked_cedula(cedula),
        "title": f"{template.title} para {context}",
        "context": context,
        "purpose": template.purpose,
        "board": BOARD_NAME,
        "board_wokwi_id": BOARD_WOKWI_ID,
        "sensor": sensor,
        "actuator": actuator,
        "display": display,
        "variable": variable,
        "threshold": threshold,
        "unit": template.unit,
        "condition": template.condition,
        "hysteresis": hysteresis,
        "sample_ms": sample_ms,
        "publish_seconds": publish_seconds,
        "retry_seconds": retry_seconds,
        "alert_confirmations": alert_confirmations,
        "topic_root": topic_root,
        "authorized_uid": authorized_uid,
        "low_test": template.low_test,
        "high_test": template.high_test,
        "is_rfid": template.sensor == "mfrc522",
        "template_key": template.key,
    }
    return assignment


def _styles() -> dict[str, ParagraphStyle]:
    base = getSampleStyleSheet()
    return {
        "title": ParagraphStyle(
            "ProjectTitle",
            parent=base["Title"],
            fontName="Helvetica-Bold",
            fontSize=22,
            leading=26,
            textColor=NAVY,
            alignment=TA_CENTER,
            spaceAfter=14,
        ),
        "subtitle": ParagraphStyle(
            "ProjectSubtitle",
            parent=base["Normal"],
            fontName="Helvetica",
            fontSize=11,
            leading=15,
            textColor=BLUE,
            alignment=TA_CENTER,
            spaceAfter=12,
        ),
        "h1": ParagraphStyle(
            "SectionHeading",
            parent=base["Heading1"],
            fontName="Helvetica-Bold",
            fontSize=15,
            leading=18,
            textColor=NAVY,
            spaceBefore=6,
            spaceAfter=6,
            keepWithNext=True,
        ),
        "h2": ParagraphStyle(
            "SubsectionHeading",
            parent=base["Heading2"],
            fontName="Helvetica-Bold",
            fontSize=11.5,
            leading=14,
            textColor=ORANGE,
            spaceBefore=6,
            spaceAfter=5,
            keepWithNext=True,
        ),
        "body": ParagraphStyle(
            "Body",
            parent=base["BodyText"],
            fontName="Helvetica",
            fontSize=9.5,
            leading=13.5,
            textColor=BLACK,
            alignment=TA_LEFT,
            spaceAfter=5,
        ),
        "table_header": ParagraphStyle(
            "TableHeader",
            parent=base["BodyText"],
            fontName="Helvetica-Bold",
            fontSize=9.2,
            leading=12,
            textColor=WHITE,
            alignment=TA_LEFT,
        ),
        "small": ParagraphStyle(
            "Small",
            parent=base["BodyText"],
            fontName="Helvetica",
            fontSize=8.2,
            leading=11,
            textColor=MID_GRAY,
        ),
        "bullet": ParagraphStyle(
            "Bullet",
            parent=base["BodyText"],
            fontName="Helvetica",
            fontSize=9.3,
            leading=13,
            leftIndent=13,
            firstLineIndent=-8,
            bulletIndent=2,
            textColor=BLACK,
            spaceAfter=2,
        ),
        "callout": ParagraphStyle(
            "Callout",
            parent=base["BodyText"],
            fontName="Helvetica-Bold",
            fontSize=10,
            leading=14,
            textColor=NAVY,
            alignment=TA_CENTER,
        ),
    }


def _plain_paragraph(text: Any, style: ParagraphStyle) -> Paragraph:
    escaped = html.escape(str(text)).replace("\n", "<br/>")
    return Paragraph(escaped, style)


def _bullet_list(items: Sequence[str], style: ParagraphStyle) -> list[Paragraph]:
    return [
        Paragraph(f"- {html.escape(str(item))}", style)
        for item in items
    ]


def _info_table(
    rows: Sequence[tuple[str, Any]],
    styles: dict[str, ParagraphStyle],
    widths: tuple[float, float] = (4.1 * cm, 12.0 * cm),
) -> Table:
    data = [
        [
            Paragraph(f"<b>{html.escape(label)}</b>", styles["body"]),
            _plain_paragraph(value, styles["body"]),
        ]
        for label, value in rows
    ]
    table = Table(data, colWidths=list(widths), hAlign="LEFT", repeatRows=0)
    table.setStyle(
        TableStyle(
            [
                ("BACKGROUND", (0, 0), (0, -1), PALE_BLUE),
                ("TEXTCOLOR", (0, 0), (-1, -1), BLACK),
                ("VALIGN", (0, 0), (-1, -1), "TOP"),
                ("GRID", (0, 0), (-1, -1), 0.35, colors.HexColor("#C8D6E0")),
                ("LEFTPADDING", (0, 0), (-1, -1), 7),
                ("RIGHTPADDING", (0, 0), (-1, -1), 7),
                ("TOPPADDING", (0, 0), (-1, -1), 6),
                ("BOTTOMPADDING", (0, 0), (-1, -1), 6),
            ]
        )
    )
    return table


def _page_decorator(project_code: str):
    def draw(canvas: Any, document: Any) -> None:
        canvas.saveState()
        width, height = letter
        canvas.setFillColor(NAVY)
        canvas.rect(0, height - 0.55 * cm, width, 0.55 * cm, stroke=0, fill=1)
        canvas.setFillColor(MID_GRAY)
        canvas.setFont("Helvetica", 7.5)
        footer_left = f"{COURSE_NAME} | {project_code}"
        canvas.drawString(1.7 * cm, 0.72 * cm, footer_left)
        page_text = f"Pagina {document.page}"
        canvas.drawRightString(width - 1.7 * cm, 0.72 * cm, page_text)
        canvas.restoreState()

    return draw


def generate_pdf(cedula: str, output_directory: str | Path | None = None) -> Path:
    """Generate the assigned PDF and return its absolute path."""
    normalized = normalize_cedula(cedula)
    assignment = build_assignment(normalized)

    destination = Path(output_directory or Path.cwd()).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    pdf_path = destination / f"proyecto_{assignment['project_code'].lower()}.pdf"

    document = SimpleDocTemplate(
        str(pdf_path),
        pagesize=letter,
        rightMargin=1.7 * cm,
        leftMargin=1.7 * cm,
        topMargin=1.35 * cm,
        bottomMargin=1.35 * cm,
        title=f"Proyecto integrador {assignment['project_code']}",
        author="Curso de Internet de las Cosas",
        subject="Asignacion individual de proyecto integrador con ESP32 y Wokwi",
    )
    styles = _styles()
    story: list[Any] = []

    story.extend(
        [
            Spacer(1, 0.55 * cm),
            _plain_paragraph("PROYECTO INTEGRADOR INDIVIDUAL", styles["subtitle"]),
            _plain_paragraph(assignment["title"], styles["title"]),
            _plain_paragraph(
                "ESP32, Wokwi, PlatformIO y conectividad IoT",
                styles["subtitle"],
            ),
            Spacer(1, 0.25 * cm),
        ]
    )

    identity_table = _info_table(
        (
            ("Codigo del proyecto", assignment["project_code"]),
            ("Cedula", assignment["masked_cedula"]),
            ("Version de asignacion", COURSE_VERSION),
            ("Modalidad", "Individual"),
        ),
        styles,
    )
    story.extend([identity_table, Spacer(1, 0.35 * cm)])

    story.append(_plain_paragraph("Contexto y uso del documento", styles["h1"]))
    usage_context = Table(
        [[_plain_paragraph(
            "Este documento entrega exclusivamente los insumos individuales asociados al "
            "proyecto y esta organizado en tres bloques: actividad 1, actividad 2 y actividad 3. "
            "Cada bloque contiene datos, componentes, parametros y referencias asignadas. "
            "La actividad 2 se asocia con una plataforma en la nube y la actividad 3 con "
            "un broker Mosquitto ejecutado localmente. "
            "Los requerimientos, las acciones, los productos, las evidencias, los criterios "
            "de evaluacion y las instrucciones de entrega no forman parte de este PDF; se "
            "publicaran paulatinamente en la plataforma del curso. Cada publicacion identificara "
            "el bloque de insumos correspondiente. El documento conserva la misma asignacion "
            "individual durante todo el curso.",
            styles["body"],
        )]],
        colWidths=[16.1 * cm],
    )
    usage_context.setStyle(
        TableStyle(
            [
                ("BACKGROUND", (0, 0), (-1, -1), PALE_BLUE),
                ("BOX", (0, 0), (-1, -1), 0.7, BLUE),
                ("LEFTPADDING", (0, 0), (-1, -1), 12),
                ("RIGHTPADDING", (0, 0), (-1, -1), 12),
                ("TOPPADDING", (0, 0), (-1, -1), 10),
                ("BOTTOMPADDING", (0, 0), (-1, -1), 10),
            ]
        )
    )
    story.extend([usage_context, Spacer(1, 0.3 * cm)])

    def activity_intro(text: str) -> Table:
        box = Table(
            [[_plain_paragraph(text, styles["callout"])]],
            colWidths=[16.1 * cm],
        )
        box.setStyle(
            TableStyle(
                [
                    ("BACKGROUND", (0, 0), (-1, -1), PALE_BLUE),
                    ("BOX", (0, 0), (-1, -1), 0.7, BLUE),
                    ("LEFTPADDING", (0, 0), (-1, -1), 12),
                    ("RIGHTPADDING", (0, 0), (-1, -1), 12),
                    ("TOPPADDING", (0, 0), (-1, -1), 9),
                    ("BOTTOMPADDING", (0, 0), (-1, -1), 9),
                ]
            )
        )
        return box

    def neutral_reference(text: str) -> str:
        result = text.strip()
        for prefix in ("Simular ", "Generar ", "Presentar "):
            if result.startswith(prefix):
                result = result[len(prefix):]
                break
        result = result.replace(
            " y comprobar el bloqueo.",
            "; bloqueo esperado tras el tercer rechazo.",
        )
        return result[:1].upper() + result[1:]

    threshold_label = "Tiempo de bloqueo" if assignment["is_rfid"] else "Umbral principal"
    threshold_value = f"{assignment['threshold']} {assignment['unit']}"

    scenario_rows = (
        ("Contexto", assignment["context"]),
        ("Proposito", assignment["purpose"]),
        ("Variable principal", assignment["variable"]),
        ("Regla asignada", assignment["condition"]),
    )

    components_rows = (
        (
            "Microcontrolador",
            f"{assignment['board']} - identificador Wokwi: {assignment['board_wokwi_id']}",
        ),
        (
            "Sensor",
            f"{assignment['sensor']['name']} - {assignment['sensor']['interface']} - identificador Wokwi: {assignment['sensor']['wokwi_id']}",
        ),
        (
            "Actuador",
            f"{assignment['actuator']['name']} - identificador Wokwi: {assignment['actuator']['wokwi_id']}",
        ),
        (
            "Interfaz local",
            f"{assignment['display']['name']} - identificador Wokwi: {assignment['display']['wokwi_id']}",
        ),
    )

    activity_one_parameters: list[tuple[str, Any]] = [
        (threshold_label, threshold_value),
        ("Intervalo de muestreo", f"{assignment['sample_ms']} ms"),
        ("Confirmacion de alarma", f"{assignment['alert_confirmations']} lecturas consecutivas"),
        ("Margen de retorno", f"{assignment['hysteresis']} unidades respecto al umbral"),
        ("Estado seguro", assignment["actuator"]["safe_state"]),
    ]
    if assignment["is_rfid"]:
        activity_one_parameters.insert(1, ("UID autorizado", assignment["authorized_uid"]))

    complete_parameters: list[tuple[str, Any]] = [
        (threshold_label, threshold_value),
        ("Intervalo de muestreo", f"{assignment['sample_ms']} ms"),
        ("Periodo de publicacion", f"{assignment['publish_seconds']} segundos"),
        ("Intervalo de reconexion", f"{assignment['retry_seconds']} segundos"),
        ("Confirmacion de alarma", f"{assignment['alert_confirmations']} lecturas consecutivas"),
        ("Margen de retorno", f"{assignment['hysteresis']} unidades respecto al umbral"),
        ("Comandos remotos", assignment["actuator"]["commands"]),
        ("Estado seguro", assignment["actuator"]["safe_state"]),
    ]
    if assignment["is_rfid"]:
        complete_parameters.insert(1, ("UID autorizado", assignment["authorized_uid"]))

    cloud_rows = (
        ("Destino de datos", "Plataforma en la nube"),
        ("Servicio especifico", "Definido en el enunciado publicado para la actividad 2"),
    )

    local_mqtt_rows = (
        ("Broker", "Eclipse Mosquitto"),
        ("Ubicacion", "Entorno local"),
        ("Tema base", assignment["topic_root"]),
        ("Telemetria", f"{assignment['topic_root']}/telemetry"),
        ("Estado", f"{assignment['topic_root']}/status"),
        ("Comandos", f"{assignment['topic_root']}/command"),
        ("Alertas", f"{assignment['topic_root']}/alert"),
    )

    reference_rows = (
        ("Condicion de referencia A", neutral_reference(assignment["low_test"])),
        ("Condicion de referencia B", neutral_reference(assignment["high_test"])),
        ("Dato no valido de referencia", "Lectura fuera del rango admisible o valor no numerico."),
        ("Duracion de interrupcion de red", "20 segundos"),
        ("Intervalo de reconexion", f"{assignment['retry_seconds']} segundos"),
        ("Comandos asignados", assignment["actuator"]["commands"]),
        ("Comando no reconocido de referencia", "COMANDO_DESCONOCIDO"),
        ("Estado seguro asignado", assignment["actuator"]["safe_state"]),
    )

    json_example = (
        "{<br/>"
        f'&nbsp;&nbsp;"device_id": "{html.escape(assignment["project_code"])}",<br/>'
        f'&nbsp;&nbsp;"variable": "{html.escape(assignment["variable"])}",<br/>'
        '&nbsp;&nbsp;"value": 0.0,<br/>'
        f'&nbsp;&nbsp;"unit": "{html.escape(assignment["unit"])}",<br/>'
        '&nbsp;&nbsp;"mode": "AUTO",<br/>'
        '&nbsp;&nbsp;"alarm": false,<br/>'
        '&nbsp;&nbsp;"sequence": 1<br/>'
        "}"
    )
    code_style = ParagraphStyle(
        "Code",
        parent=styles["body"],
        fontName="Courier",
        fontSize=8.2,
        leading=11,
        leftIndent=8,
        textColor=BLACK,
    )

    def telemetry_code_box() -> Table:
        box = Table([[Paragraph(json_example, code_style)]], colWidths=[16.1 * cm])
        box.setStyle(
            TableStyle(
                [
                    ("BACKGROUND", (0, 0), (-1, -1), LIGHT_GRAY),
                    ("BOX", (0, 0), (-1, -1), 0.5, colors.HexColor("#BCC6CC")),
                    ("LEFTPADDING", (0, 0), (-1, -1), 9),
                    ("RIGHTPADDING", (0, 0), (-1, -1), 9),
                    ("TOPPADDING", (0, 0), (-1, -1), 8),
                    ("BOTTOMPADDING", (0, 0), (-1, -1), 8),
                ]
            )
        )
        return box

    story.append(_plain_paragraph("1. Insumos para la actividad 1", styles["h1"]))
    story.extend(
        [
            activity_intro("Ficha de insumos individuales - Actividad 1 - 25 %."),
            Spacer(1, 0.2 * cm),
        ]
    )
    story.append(_plain_paragraph("1.1. Datos del escenario", styles["h2"]))
    story.extend([_info_table(scenario_rows, styles), Spacer(1, 0.15 * cm)])
    story.append(_plain_paragraph("1.2. Componentes asignados", styles["h2"]))
    story.extend([_info_table(components_rows, styles), Spacer(1, 0.15 * cm)])
    story.append(_plain_paragraph("1.3. Parametros individuales", styles["h2"]))
    story.extend([_info_table(tuple(activity_one_parameters), styles), Spacer(1, 0.15 * cm)])

    story.append(_plain_paragraph("2. Insumos para la actividad 2", styles["h1"]))
    story.extend(
        [
            activity_intro("Ficha de insumos individuales - Actividad 2 - 35 % - Plataforma en la nube."),
            Spacer(1, 0.2 * cm),
        ]
    )
    story.append(_plain_paragraph("2.1. Datos del escenario", styles["h2"]))
    story.extend([_info_table(scenario_rows, styles), Spacer(1, 0.15 * cm)])
    story.append(_plain_paragraph("2.2. Componentes asignados", styles["h2"]))
    story.extend([_info_table(components_rows, styles), Spacer(1, 0.15 * cm)])
    story.append(_plain_paragraph("2.3. Parametros individuales", styles["h2"]))
    story.extend([_info_table(tuple(complete_parameters), styles), Spacer(1, 0.15 * cm)])
    story.append(_plain_paragraph("2.4. Destino de datos", styles["h2"]))
    story.extend([_info_table(cloud_rows, styles), Spacer(1, 0.15 * cm)])
    story.append(_plain_paragraph("2.5. Condiciones de referencia", styles["h2"]))
    story.extend([_info_table(reference_rows, styles), Spacer(1, 0.25 * cm)])

    story.append(CondPageBreak(11.0 * cm))
    story.append(_plain_paragraph("3. Insumos para la actividad 3", styles["h1"]))
    story.extend(
        [
            activity_intro("Ficha de insumos individuales - Actividad 3 - 40 % - Mosquitto local."),
            Spacer(1, 0.2 * cm),
        ]
    )
    story.append(_plain_paragraph("3.1. Datos del escenario", styles["h2"]))
    story.extend([_info_table(scenario_rows, styles), Spacer(1, 0.15 * cm)])
    story.append(_plain_paragraph("3.2. Componentes asignados", styles["h2"]))
    story.extend([_info_table(components_rows, styles), Spacer(1, 0.15 * cm)])
    story.append(_plain_paragraph("3.3. Parametros individuales", styles["h2"]))
    story.extend([_info_table(tuple(complete_parameters), styles), Spacer(1, 0.15 * cm)])
    story.append(_plain_paragraph("3.4. Broker local Mosquitto y temas MQTT", styles["h2"]))
    story.extend([_info_table(local_mqtt_rows, styles), Spacer(1, 0.15 * cm)])
    story.append(_plain_paragraph("3.5. Formato de datos asignado", styles["h2"]))
    story.extend([telemetry_code_box(), Spacer(1, 0.15 * cm)])
    story.append(_plain_paragraph("3.6. Condiciones de referencia", styles["h2"]))
    story.extend([_info_table(reference_rows, styles), Spacer(1, 0.25 * cm)])

    decorator = _page_decorator(assignment["project_code"])
    document.build(story, onFirstPage=decorator, onLaterPages=decorator)
    return pdf_path


def _download_in_colab(pdf_path: Path) -> bool:
    try:
        from google.colab import files  # type: ignore
    except ModuleNotFoundError:
        return False
    files.download(str(pdf_path))
    return True


def main() -> None:
    print("=" * 64)
    print("GENERADOR DE PROYECTOS INTEGRADORES - INTERNET DE LAS COSAS")
    print("=" * 64)
    print("La cedula se usa solo para calcular la asignacion y no se almacena.")

    raw_cedula = getpass("Ingrese su cedula: ")
    try:
        cedula = normalize_cedula(raw_cedula)
        output_dir = Path("/content") if Path("/content").exists() else Path.cwd()
        pdf_path = generate_pdf(cedula, output_dir)
    except ValueError as exc:
        raise SystemExit(f"Error: {exc}") from exc
    finally:
        raw_cedula = ""
        if "cedula" in locals():
            cedula = ""

    print(f"PDF generado correctamente: {pdf_path.name}")
    if _download_in_colab(pdf_path):
        print("La descarga automatica fue iniciada por Colab.")
    else:
        print(f"Archivo disponible en: {pdf_path}")


if __name__ == "__main__":
    main()


Instalando la biblioteca necesaria para crear el PDF...
GENERADOR DE PROYECTOS INTEGRADORES - INTERNET DE LAS COSAS
La cedula se usa solo para calcular la asignacion y no se almacena.
Ingrese su cedula: ··········
PDF generado correctamente: proyecto_iot-55e637259a.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

La descarga automatica fue iniciada por Colab.
